# Sequential binary follow-up experiments v2

This notebook is the v2 follow up workflow for the sequential binary protease/cleavage experiments.

- Held-out evaluation is disabled by default and guarded by `RUN_HELDOUT_EVALUATION = False`.
- Outputs are intended for Colab GPU sessions mounted to Google Drive.
- Completed v1 results in `sequential_binary_experiments.ipynb` are not modified or overwritten.
- The current default stage is the optional frozen ESM2 150M model-capacity check: `RUN_STAGE = 'esm2_150m_check'`.

For the completed v2 stages, set `RUN_STAGE` to `model_capacity`, `lr_tuning`, or `validation_check`. For a smoke test, set `SMOKE_TEST = True`, use `RUN_STAGE = 'validation_check'`, and keep held out evaluation disabled.


In [2]:
# Optional Colab Drive mount. This is intentionally no-op outside Colab.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
from contextlib import nullcontext
from dataclasses import dataclass, asdict
import copy
import gc
import hashlib
import json
import math
import random
import re
import subprocess

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.optim import AdamW
    from torch.utils.data import Dataset, DataLoader
    TORCH_AVAILABLE = True
    TORCH_IMPORT_ERROR = None
except Exception as exc:
    torch = None
    nn = None
    F = None
    AdamW = None
    Dataset = object
    DataLoader = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc

try:
    from transformers import AutoModel, AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
    TRANSFORMERS_IMPORT_ERROR = None
except Exception as exc:
    AutoModel = None
    AutoTokenizer = None
    TRANSFORMERS_AVAILABLE = False
    TRANSFORMERS_IMPORT_ERROR = exc

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
)
from tqdm.auto import tqdm

PROJECT_ROOT_OVERRIDE = None
OUTPUT_DIR_OVERRIDE = None

PIPELINE_VERSION = 'sequential_binary_followup_v2'
MEROPS_PARSING_VERSION = 'merops_regex_class_family_v1'
NEGATIVE_GENERATION_VERSION = 'partition_local_cross_family_v1'

RUN_STAGE = 'esm2_150m_check'  # model_capacity, lr_tuning, validation_check, esm2_150m_check, all
SMOKE_TEST = False
RUN_HELDOUT_EVALUATION = False
ALLOW_OVERWRITE_RUNS = False
REUSE_EXISTING_COMPLETED_RUNS = True
INCLUDE_LR_1E4_SANITY_CHECK = False
CACHE_FROZEN_ENCODERS = True

DEFAULT_SPLIT_SEED = 42
VALIDATION_CHECK_SPLIT_SEEDS = [42, 43, 44]
SMOKE_VALIDATION_CHECK_SPLIT_SEEDS = [42]
TRAINING_SEED = 42
PAIR_GENERATION_SEED = 4242

EPOCHS = 1 if SMOKE_TEST else 20
VALID_FRACTION = 0.15
NEGATIVES_PER_POSITIVE = 1
THRESHOLDS = np.linspace(0.05, 0.95, 19)

MAX_LEN_PROTEASE = 1022
MAX_LEN_SITE = 16
DROPOUT = 0.20
HIDDEN_DIM = 256
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

LR_UNFROZEN_DEFAULT = 1e-5
LR_TUNING_VALUES = [1e-7]
LR_1E4_SANITY_VALUE = 1e-4

MODEL_150M = 'facebook/esm2_t30_150M_UR50D'
EPOCHS_150M = 9
LRS_150M_FROZEN = [1e-7] #1e-5

MODEL_SPECS = {
    'facebook/esm2_t6_8M_UR50D': {
        'short_name': 'esm2_8m',
        'frozen_batch_size': 4,
        'frozen_grad_accum': 1,
        'unfrozen_batch_size': 4,
        'unfrozen_grad_accum': 1,
    },
    'facebook/esm2_t12_35M_UR50D': {
        'short_name': 'esm2_35m',
        'frozen_batch_size': 2,
        'frozen_grad_accum': 2,
        'unfrozen_batch_size': 1,
        'unfrozen_grad_accum': 4,
    },
    'facebook/esm2_t30_150M_UR50D': {
        'short_name': 'esm2_150m',
        'frozen_batch_size': 1,
        'frozen_grad_accum': 4,
        'unfrozen_batch_size': 1,
        'unfrozen_grad_accum': 8,
    },
    'facebook/esm2_t33_650M_UR50D': {
        'short_name': 'esm2_650m',
        'frozen_batch_size': 1,
        'frozen_grad_accum': 4,
        'unfrozen_batch_size': 1,
        'unfrozen_grad_accum': 16,
    },
}

SMOKE_FAMILY_COUNT = 4
SMOKE_CODES_PER_FAMILY = 6
SMOKE_ROWS_PER_CODE = 2
SMOKE_MIN_FAMILIES = 3
SMOKE_MIN_CODES = 12

MIN_VALID_CODES_WARNING = 5
MIN_VALID_POSITIVES_WARNING = 100
MIN_STRATUM_CODES = 2
RARE_STRATUM_CODE_THRESHOLD = 3

P1_MIN_NATURAL_PER_PROTEASE = 10
P1_NONPHYSIO_CAP_PER_CLUSTER = 2000
NATURAL_TYPES = {'physiological', 'pathological'}
AMINO_ACIDS = list('ARNDCQEGHILKMFPSTWYV')
AA_TO_P1_CLUSTER = {
    'A': 1, 'V': 1, 'I': 1, 'L': 1, 'M': 1,
    'F': 2, 'Y': 2, 'W': 2,
    'S': 3, 'T': 3, 'N': 3, 'Q': 3,
    'D': 4, 'E': 4,
    'K': 5, 'R': 5,
    'H': 6,
    'G': 7,
    'P': 8,
    'C': 9,
}

INTERMEDIATE_STAGES_WITHOUT_HELDOUT = {'model_capacity', 'lr_tuning', 'validation_check', 'esm2_150m_check', 'all'}
if RUN_HELDOUT_EVALUATION and RUN_STAGE in INTERMEDIATE_STAGES_WITHOUT_HELDOUT:
    raise ValueError('Held-out evaluation must stay disabled for intermediate experiment stages.')

print(f'Pipeline version: {PIPELINE_VERSION}')
print(f'Smoke test:       {SMOKE_TEST}')
print(f'Run stage:        {RUN_STAGE}')
print(f'Held-out enabled: {RUN_HELDOUT_EVALUATION}')
if RUN_STAGE == 'esm2_150m_check':
    print('150M architecture: frozen_esm')
    print(f'150M epochs:      {EPOCHS_150M}')
    print(f'150M learning rate: {LRS_150M_FROZEN}')
print(f'Torch available:  {TORCH_AVAILABLE}')
print(f'Transformers:     {TRANSFORMERS_AVAILABLE}')


Pipeline version: sequential_binary_followup_v2
Smoke test:       False
Run stage:        esm2_150m_check
Held-out enabled: False
150M architecture: frozen_esm
150M epochs:      9
150M learning rate: [1e-07]
Torch available:  True
Transformers:     True


In [4]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def find_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE)
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/deep-learning-proteases'),
        Path('/content/drive/MyDrive/deep-learning-proteases'),
        Path('/content/drive/MyDrive/MEIN40430 Materials/deep-learning-proteases'),
    ]
    for candidate in candidates:
        if (candidate / 'processed_data' / 'training.csv').exists():
            return candidate
    search_roots = [Path.cwd(), Path('/content')]
    if Path('/content/drive/MyDrive').exists():
        search_roots.append(Path('/content/drive/MyDrive'))
    for root in search_roots:
        try:
            matches = list(root.glob('**/processed_data/training.csv'))
        except Exception:
            matches = []
        if matches:
            return matches[0].parents[1]
    raise FileNotFoundError('Could not find processed_data/training.csv.')


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str | None:
    if path is None or not Path(path).exists():
        return None
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def make_json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [make_json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def canonical_json(value) -> str:
    return json.dumps(make_json_safe(value), sort_keys=True, separators=(',', ':'))


def short_hash(value, length: int = 8) -> str:
    text = canonical_json(value) if not isinstance(value, str) else value
    return hashlib.sha256(text.encode('utf-8')).hexdigest()[:length]


def write_json(path: Path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(make_json_safe(value), indent=2, sort_keys=True), encoding='utf-8')


def git_commit_hash() -> str | None:
    try:
        result = subprocess.run(
            ['git', 'rev-parse', 'HEAD'],
            cwd=PROJECT_ROOT,
            text=True,
            capture_output=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception:
        return None


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'processed_data'
OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE) if OUTPUT_DIR_OVERRIDE is not None else PROJECT_ROOT / 'outputs' / 'sequential_binary_followup_v2'
MANIFEST_DIR = OUTPUT_DIR / '_manifests'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA_DIR / 'training.csv'
TEST_SINGLETON_CSV = DATA_DIR / 'test_1_sample.csv'
TEST_LOW_SAMPLE_CSV = DATA_DIR / 'test_2_to_10_samples.csv'
TRAIN_8MER_CSV = DATA_DIR / 'training_8mer_clustering.csv'
ROOT_8MER_CSV = PROJECT_ROOT / 'training_8mer_clustering.csv'

DATASET_HASHES = {
    'training_csv': file_sha256(TRAIN_CSV),
    'test_1_sample_csv': file_sha256(TEST_SINGLETON_CSV),
    'test_2_to_10_samples_csv': file_sha256(TEST_LOW_SAMPLE_CSV),
    'training_8mer_clustering_csv': file_sha256(TRAIN_8MER_CSV if TRAIN_8MER_CSV.exists() else ROOT_8MER_CSV),
}

print(f'Project root: {PROJECT_ROOT}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'Manifest dir: {MANIFEST_DIR}')
print(f'Git commit:   {git_commit_hash()}')


Project root: /content/drive/MyDrive/MEIN40430 Materials/deep-learning-proteases
Output dir:   /content/drive/MyDrive/MEIN40430 Materials/deep-learning-proteases/outputs/sequential_binary_followup_v2
Manifest dir: /content/drive/MyDrive/MEIN40430 Materials/deep-learning-proteases/outputs/sequential_binary_followup_v2/_manifests
Git commit:   1fb5b65f28b79cb3a8b875fe297be82b6837154d


In [5]:
MEROPS_CODE_RE = re.compile(r'^\s*([A-Za-z])([0-9A-Za-z]+)(?:\..*)?\s*$')
MEROPS_PARSE_METHOD = 'regex ^([A-Za-z])([0-9A-Za-z]+)(?:\\..*)?$; class=group1, family=group1+group2 before decimal'


def normalize_code(value) -> str:
    return str(value).strip()


def normalize_site(value) -> str:
    return str(value).strip().upper().replace(' ', '')


def normalize_pair(code, site):
    return (normalize_code(code), normalize_site(site))


def clean_site(site: str) -> str:
    return normalize_site(site)


def clean_for_esm(seq: str) -> str:
    seq = str(seq).upper().replace(' ', '')
    return seq.replace('-', 'X')


def parse_merops_code(value) -> dict:
    protease_code = normalize_code(value)
    if protease_code == '' or protease_code.lower() in {'nan', 'none', 'null'}:
        return {
            'protease_code': protease_code,
            'merops_class': 'unparsed_merops',
            'merops_family': 'unparsed_merops',
            'merops_parse_status': 'missing_or_empty',
        }
    match = MEROPS_CODE_RE.match(protease_code)
    if match is None:
        return {
            'protease_code': protease_code,
            'merops_class': 'unparsed_merops',
            'merops_family': 'unparsed_merops',
            'merops_parse_status': 'regex_no_match',
        }
    merops_class = match.group(1).upper()
    family_suffix = match.group(2).upper()
    return {
        'protease_code': protease_code,
        'merops_class': merops_class,
        'merops_family': f'{merops_class}{family_suffix}',
        'merops_parse_status': 'parsed_regex_class_family_before_decimal',
    }


def add_merops_fields(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    parsed = out['code'].map(parse_merops_code).apply(pd.Series)
    for column in parsed.columns:
        out[column] = parsed[column]
    out['code'] = out['protease_code']
    out['family'] = out['merops_family']
    return out


def merops_parse_examples(df: pd.DataFrame, n: int = 12) -> list[dict]:
    cols = ['protease_code', 'merops_class', 'merops_family', 'merops_parse_status']
    return df[cols].drop_duplicates().head(n).to_dict(orient='records')


def class_family_distribution(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    grouped = df.groupby(['merops_class', 'merops_family'], dropna=False).agg(
        rows=('protease_code', 'size'),
        protease_codes=('protease_code', 'nunique'),
        unique_sites=('site', 'nunique'),
    ).reset_index()
    grouped.insert(0, 'partition', prefix)
    return grouped.sort_values(['partition', 'merops_class', 'merops_family']).reset_index(drop=True)


def load_processed_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {'code', 'site', 'protease', 'cleavage_type'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    df = df.dropna(subset=['code', 'site', 'protease']).copy()
    df['code'] = df['code'].map(normalize_code)
    df['site'] = df['site'].map(normalize_site)
    df['protease'] = df['protease'].astype(str).str.strip()
    df['cleavage_type'] = df['cleavage_type'].astype(str).str.strip()
    df = add_merops_fields(df)
    df['source_file'] = str(path)
    df['source_row_number'] = np.arange(len(df), dtype=int)
    df['source_row_id'] = path.stem + ':' + df['source_row_number'].astype(str)
    df['source_positive_count'] = df.groupby('protease_code')['protease_code'].transform('size').astype(int)
    return df


def find_p1_csv() -> Path | None:
    candidates = [
        DATA_DIR / 'training_p1_clustering.csv',
        DATA_DIR / 'training_p1_clustered.csv',
        DATA_DIR / 'training_p1_site_clustering.csv',
        DATA_DIR / 'training_1mer_clustering.csv',
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = sorted(DATA_DIR.glob('*p1*clustering*.csv')) + sorted(DATA_DIR.glob('*1mer*clustering*.csv'))
    return matches[0] if matches else None


def p1_residue(site: str) -> str:
    site = clean_site(site)
    return site[3] if len(site) >= 4 else 'X'


def add_p1_cluster(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['p1_aa'] = out['site'].map(p1_residue)
    out['p1_cluster'] = out['p1_aa'].map(AA_TO_P1_CLUSTER).fillna(0).astype(int)
    return out


def select_p1_positive_rows(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    df = add_p1_cluster(df)
    natural_mask = df['cleavage_type'].isin(NATURAL_TYPES)
    natural_counts = df[natural_mask].groupby('protease_code').size()
    valid_codes = natural_counts[natural_counts >= P1_MIN_NATURAL_PER_PROTEASE].index.tolist()
    natural_samples = df[df['protease_code'].isin(valid_codes) & natural_mask].copy()
    available_nonphysio = df[df['protease_code'].isin(valid_codes) & (df['cleavage_type'] == 'non-physiological')].copy()
    selected = [natural_samples]
    for cluster_id in sorted(c for c in df['p1_cluster'].dropna().unique() if c != 0):
        cluster_rows = available_nonphysio[available_nonphysio['p1_cluster'] == cluster_id]
        n_take = min(P1_NONPHYSIO_CAP_PER_CLUSTER, len(cluster_rows))
        if n_take > 0:
            selected.append(cluster_rows.sample(n=n_take, random_state=seed))
    out = pd.concat(selected, ignore_index=True)
    subset = ['protease_code', 'site', 'cleavageID'] if 'cleavageID' in out.columns else ['protease_code', 'site']
    return out.drop_duplicates(subset=subset).reset_index(drop=True)


def select_code_balanced_smoke_rows(df: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    code_table = df.groupby('protease_code').agg(
        merops_family=('merops_family', 'first'),
        merops_class=('merops_class', 'first'),
        source_positive_count=('source_positive_count', 'first'),
        available_rows=('protease_code', 'size'),
    ).reset_index()
    eligible_families = (
        code_table[code_table['merops_family'] != 'unparsed_merops']
        .groupby('merops_family')['protease_code']
        .nunique()
        .sort_values(ascending=False)
    )
    eligible_families = eligible_families[eligible_families >= SMOKE_CODES_PER_FAMILY]
    if len(eligible_families) < SMOKE_MIN_FAMILIES:
        raise ValueError('Not enough MEROPS families/codes for a code-balanced smoke sample.')
    chosen_families = eligible_families.index[:SMOKE_FAMILY_COUNT].tolist()
    selected_frames = []
    selected_codes = []
    for family in chosen_families:
        family_codes = sorted(code_table.loc[code_table['merops_family'] == family, 'protease_code'].tolist())
        if len(family_codes) > SMOKE_CODES_PER_FAMILY:
            family_codes = sorted(rng.choice(family_codes, size=SMOKE_CODES_PER_FAMILY, replace=False).tolist())
        selected_codes.extend(family_codes)
        for code in family_codes:
            rows = df[df['protease_code'] == code]
            n_take = min(SMOKE_ROWS_PER_CODE, len(rows))
            selected_frames.append(rows.sample(n=n_take, random_state=seed + len(selected_frames)))
    out = pd.concat(selected_frames, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    n_families = out['merops_family'].nunique()
    n_codes = out['protease_code'].nunique()
    if n_families < SMOKE_MIN_FAMILIES or n_codes < SMOKE_MIN_CODES:
        raise ValueError(f'Smoke sample too small: {n_families} families, {n_codes} codes.')
    info = {
        'smoke_sampling': 'code_balanced_by_merops_family',
        'smoke_seed': seed,
        'requested_family_count': SMOKE_FAMILY_COUNT,
        'requested_codes_per_family': SMOKE_CODES_PER_FAMILY,
        'requested_rows_per_code': SMOKE_ROWS_PER_CODE,
        'selected_families': chosen_families,
        'selected_codes': selected_codes,
        'rows': int(len(out)),
        'protease_codes': int(n_codes),
        'merops_families': int(n_families),
    }
    return out, info


test_singleton_raw = load_processed_csv(TEST_SINGLETON_CSV)
test_low_sample_raw = load_processed_csv(TEST_LOW_SAMPLE_CSV)
HELDOUT_CODES = set(test_singleton_raw['protease_code']).union(set(test_low_sample_raw['protease_code']))
P1_CSV = find_p1_csv()

print(f'P1 CSV: {P1_CSV if P1_CSV else "not found; internal P1 selection available"}')
print(f'8-mer CSV exists: {(TRAIN_8MER_CSV.exists() or ROOT_8MER_CSV.exists())}')
print(f'Held-out protease codes: {len(HELDOUT_CODES):,}')
print('MEROPS parse examples:')
display(pd.DataFrame(merops_parse_examples(load_processed_csv(TRAIN_CSV))).head(12))


P1 CSV: /content/drive/MyDrive/MEIN40430 Materials/deep-learning-proteases/processed_data/training_p1_clustering.csv
8-mer CSV exists: True
Held-out protease codes: 557
MEROPS parse examples:


,protease_code,merops_class,merops_family,merops_parse_status
0,S01.151,S,S01,parsed_regex_class_family_before_decimal
1,S01.154,S,S01,parsed_regex_class_family_before_decimal
2,S01.146,S,S01,parsed_regex_class_family_before_decimal
3,S01.223,S,S01,parsed_regex_class_family_before_decimal
4,S01.163,S,S01,parsed_regex_class_family_before_decimal
5,S01.170,S,S01,parsed_regex_class_family_before_decimal
6,S01.191,S,S01,parsed_regex_class_family_before_decimal
7,S01.217,S,S01,parsed_regex_class_family_before_decimal
8,S01.233,S,S01,parsed_regex_class_family_before_decimal
9,M10.008,M,M10,parsed_regex_class_family_before_decimal


In [6]:
positive_source_cache = {}


def source_path_for_name(source_name: str) -> Path | None:
    if source_name == 'no_clustering':
        return TRAIN_CSV
    if source_name == 'p1_clustering':
        return P1_CSV
    if source_name == '8mer_clustering':
        if TRAIN_8MER_CSV.exists():
            return TRAIN_8MER_CSV
        return ROOT_8MER_CSV if ROOT_8MER_CSV.exists() else TRAIN_8MER_CSV
    raise ValueError(f'Unknown source: {source_name}')


def load_positive_source(source_name: str, seed: int = 42) -> tuple[pd.DataFrame, dict]:
    cache_key = (source_name, seed, SMOKE_TEST)
    if cache_key in positive_source_cache:
        frame, info = positive_source_cache[cache_key]
        return frame.copy(), copy.deepcopy(info)

    source_path = source_path_for_name(source_name)
    used_internal_p1 = False
    if source_name == 'p1_clustering' and source_path is None:
        base = load_processed_csv(TRAIN_CSV)
        df = select_p1_positive_rows(base, seed)
        used_internal_p1 = True
    else:
        if source_path is None or not source_path.exists():
            raise FileNotFoundError(f'Missing {source_path}. Add the required clustering CSV first.')
        df = load_processed_csv(source_path)

    rows_before = len(df)
    codes_before = int(df['protease_code'].nunique())
    df = df[~df['protease_code'].isin(HELDOUT_CODES)].copy().reset_index(drop=True)
    rows_after_heldout = len(df)
    codes_after_heldout = int(df['protease_code'].nunique())

    smoke_info = None
    if SMOKE_TEST:
        df, smoke_info = select_code_balanced_smoke_rows(df, seed)

    malformed_count = int((df['merops_family'] == 'unparsed_merops').sum())
    info = {
        'source_name': source_name,
        'source_path': str(source_path) if source_path is not None else None,
        'source_sha256': file_sha256(source_path) if source_path is not None and source_path.exists() else None,
        'used_internal_p1_selection': used_internal_p1,
        'rows_before_heldout_exclusion': rows_before,
        'rows_after_heldout_exclusion': rows_after_heldout,
        'rows_removed_by_heldout_exclusion': rows_before - rows_after_heldout,
        'codes_before_heldout_exclusion': codes_before,
        'codes_after_heldout_exclusion': codes_after_heldout,
        'codes_removed_by_heldout_exclusion': codes_before - codes_after_heldout,
        'unique_sites_after_heldout_exclusion': int(df['site'].nunique()),
        'merops_parse_method': MEROPS_PARSE_METHOD,
        'merops_parsing_version': MEROPS_PARSING_VERSION,
        'merops_unparsed_rows_after_sampling': malformed_count,
        'merops_parse_examples': merops_parse_examples(df),
        'smoke_sample': smoke_info,
    }
    positive_source_cache[cache_key] = (df.copy(), copy.deepcopy(info))
    return df, info


def build_global_known_positive_pairs() -> set:
    paths = [TRAIN_CSV, TEST_SINGLETON_CSV, TEST_LOW_SAMPLE_CSV]
    if P1_CSV is not None:
        paths.append(P1_CSV)
    eight_mer_path = TRAIN_8MER_CSV if TRAIN_8MER_CSV.exists() else ROOT_8MER_CSV
    if eight_mer_path.exists():
        paths.append(eight_mer_path)
    pairs = set()
    for path in paths:
        if not path.exists():
            continue
        df = pd.read_csv(path, usecols=lambda col: col in {'code', 'site'})
        if not {'code', 'site'}.issubset(df.columns):
            continue
        for code_value, site in zip(df['code'], df['site']):
            parsed = parse_merops_code(code_value)
            pairs.add(normalize_pair(parsed['protease_code'], site))
    return pairs


GLOBAL_KNOWN_POSITIVE_PAIRS = build_global_known_positive_pairs()
print(f'Known positive pairs blocked from negatives: {len(GLOBAL_KNOWN_POSITIVE_PAIRS):,}')

df_source, source_info = load_positive_source('no_clustering', seed=DEFAULT_SPLIT_SEED)
print(
    f'no_clustering: {len(df_source):,} rows, '
    f'{df_source["protease_code"].nunique():,} codes, '
    f'{df_source["merops_family"].nunique():,} MEROPS families, '
    f'{df_source["site"].nunique():,} sites'
)


Known positive pairs blocked from negatives: 59,106
no_clustering: 61,436 rows, 355 codes, 73 MEROPS families, 47,220 sites


In [7]:
def code_frequency_bin(count: int) -> str:
    count = int(count)
    if count <= 1:
        return 'n_1'
    if count <= 10:
        return 'n_2_to_10'
    if count <= 50:
        return 'n_11_to_50'
    if count <= 200:
        return 'n_51_to_200'
    return 'n_gt_200'


def make_code_table(positive_rows: pd.DataFrame) -> pd.DataFrame:
    table = positive_rows.groupby('protease_code').agg(
        merops_class=('merops_class', 'first'),
        merops_family=('merops_family', 'first'),
        positive_rows=('protease_code', 'size'),
        source_positive_count=('source_positive_count', 'first'),
        unique_sites=('site', 'nunique'),
    ).reset_index()
    table['abundance_bin'] = table['source_positive_count'].map(code_frequency_bin)
    return table.sort_values('protease_code').reset_index(drop=True)


def split_code_group_random(code_table: pd.DataFrame, seed: int) -> tuple[set[str], set[str], dict]:
    if len(code_table) < 2:
        raise ValueError('Need at least two protease codes for a split.')
    splitter = GroupShuffleSplit(n_splits=1, test_size=VALID_FRACTION, random_state=seed)
    train_idx, valid_idx = next(splitter.split(code_table, groups=code_table['protease_code']))
    train_codes = set(code_table.iloc[train_idx]['protease_code'])
    valid_codes = set(code_table.iloc[valid_idx]['protease_code'])
    metadata = {
        'requested_split_strategy': 'code_group_random',
        'actual_split_strategy': 'code_group_random',
        'fallback_used': False,
        'fallback_reason': None,
        'valid_fraction': VALID_FRACTION,
    }
    return train_codes, valid_codes, metadata


def merge_rare_strata(code_table: pd.DataFrame, base_col: str) -> tuple[pd.Series, dict]:
    strata = code_table[base_col].astype(str).copy()
    counts = strata.value_counts()
    rare_values = set(counts[counts < RARE_STRATUM_CODE_THRESHOLD].index)
    unparsed_mask = code_table['merops_family'].eq('unparsed_merops') | code_table['merops_class'].eq('unparsed_merops')
    merged = []
    for idx, value in strata.items():
        if unparsed_mask.loc[idx]:
            merged.append('unparsed_merops')
        elif value in rare_values:
            merged.append('rare_merged')
        else:
            merged.append(value)
    merged = pd.Series(merged, index=code_table.index, name=f'{base_col}_merged')
    metadata = {
        'rare_threshold_codes': RARE_STRATUM_CODE_THRESHOLD,
        'rare_values_before_merge': sorted(v for v in rare_values if v != 'unparsed_merops'),
        'rare_merged_code_count': int((merged == 'rare_merged').sum()),
        'unparsed_merops_code_count': int((merged == 'unparsed_merops').sum()),
        'strata_after_merge': merged.value_counts().sort_index().to_dict(),
    }
    return merged, metadata


def split_by_strata(code_table: pd.DataFrame, stratum_col: str, seed: int) -> tuple[set[str], set[str], dict]:
    rng = np.random.default_rng(seed)
    train_codes = set()
    valid_codes = set()
    unsplit_singleton_strata = []
    per_stratum = []
    for stratum, group in code_table.groupby(stratum_col, sort=True):
        codes = sorted(group['protease_code'].tolist())
        n_codes = len(codes)
        if n_codes < MIN_STRATUM_CODES:
            train_codes.update(codes)
            unsplit_singleton_strata.append(str(stratum))
            per_stratum.append({'stratum': str(stratum), 'codes': n_codes, 'valid_codes': 0, 'reason': 'too_few_codes_train_only'})
            continue
        n_valid = max(1, int(round(n_codes * VALID_FRACTION)))
        n_valid = min(n_valid, n_codes - 1)
        chosen_valid = set(rng.choice(codes, size=n_valid, replace=False).tolist())
        valid_codes.update(chosen_valid)
        train_codes.update(set(codes) - chosen_valid)
        per_stratum.append({'stratum': str(stratum), 'codes': n_codes, 'valid_codes': n_valid, 'reason': 'stratified_sample'})
    metadata = {
        'valid_fraction': VALID_FRACTION,
        'stratum_column': stratum_col,
        'unsplit_singleton_strata': unsplit_singleton_strata,
        'per_stratum': per_stratum,
    }
    return train_codes, valid_codes, metadata


def split_codes(positive_rows: pd.DataFrame, strategy: str, split_seed: int) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    code_table = make_code_table(positive_rows)
    requested_strategy = strategy
    if strategy == 'code_group_random':
        train_codes, valid_codes, strategy_metadata = split_code_group_random(code_table, split_seed)
    elif strategy == 'family_stratified_code':
        working = code_table.copy()
        working['split_stratum'], merge_metadata = merge_rare_strata(working, 'merops_family')
        train_codes, valid_codes, inner_metadata = split_by_strata(working, 'split_stratum', split_seed)
        strategy_metadata = {
            'requested_split_strategy': requested_strategy,
            'actual_split_strategy': 'family_stratified_code',
            'fallback_used': False,
            'fallback_reason': None,
            'merge_metadata': merge_metadata,
            **inner_metadata,
            'generalisation_note': 'Family-stratified splitting tests unseen protease codes within represented MEROPS families, not entirely unseen MEROPS families.',
        }
    elif strategy == 'family_abundance_stratified_code':
        working = code_table.copy()
        working['family_abundance_stratum_raw'] = working['merops_family'].astype(str) + '|' + working['abundance_bin'].astype(str)
        working['split_stratum'], merge_metadata = merge_rare_strata(working, 'family_abundance_stratum_raw')
        train_codes, valid_codes, inner_metadata = split_by_strata(working, 'split_stratum', split_seed)
        strategy_metadata = {
            'requested_split_strategy': requested_strategy,
            'actual_split_strategy': 'family_abundance_stratified_code',
            'fallback_used': False,
            'fallback_reason': None,
            'merge_metadata': merge_metadata,
            **inner_metadata,
            'generalisation_note': 'Family-abundance stratification balances represented MEROPS families and code-level abundance bins; it does not test entirely unseen MEROPS families.',
        }
    else:
        raise ValueError(f'Unknown validation split strategy: {strategy}')

    if len(valid_codes) == 0 or len(train_codes) == 0:
        fallback_reason = f'{strategy} produced empty train or validation code set.'
        train_codes, valid_codes, fallback_metadata = split_code_group_random(code_table, split_seed)
        strategy_metadata = {
            'requested_split_strategy': requested_strategy,
            'actual_split_strategy': 'code_group_random',
            'fallback_used': True,
            'fallback_reason': fallback_reason,
            'fallback_metadata': fallback_metadata,
        }

    train_positive = positive_rows[positive_rows['protease_code'].isin(train_codes)].copy().reset_index(drop=True)
    valid_positive = positive_rows[positive_rows['protease_code'].isin(valid_codes)].copy().reset_index(drop=True)
    assert train_codes.isdisjoint(valid_codes), 'Train and validation protease codes overlap.'
    assert train_codes.isdisjoint(HELDOUT_CODES), 'Held-out code found in training.'
    assert valid_codes.isdisjoint(HELDOUT_CODES), 'Held-out code found in validation.'

    site_overlap = sorted(set(train_positive['site']).intersection(set(valid_positive['site'])))
    train_positive_pairs = set(zip(train_positive['protease_code'], train_positive['site']))
    valid_positive_pairs = set(zip(valid_positive['protease_code'], valid_positive['site']))
    exact_positive_overlap = train_positive_pairs.intersection(valid_positive_pairs)
    assert not exact_positive_overlap, 'Exact train/validation positive protease-site pairs overlap.'

    class_family = pd.concat([
        class_family_distribution(train_positive, 'train_positive'),
        class_family_distribution(valid_positive, 'valid_positive'),
    ], ignore_index=True)

    metadata = {
        'split_seed': split_seed,
        'requested_split_strategy': requested_strategy,
        'actual_split_strategy': strategy_metadata.get('actual_split_strategy', requested_strategy),
        'strategy_metadata': strategy_metadata,
        'n_train_codes': int(len(train_codes)),
        'n_valid_codes': int(len(valid_codes)),
        'n_train_positive_rows': int(len(train_positive)),
        'n_valid_positive_rows': int(len(valid_positive)),
        'train_codes': sorted(train_codes),
        'valid_codes': sorted(valid_codes),
        'heldout_exclusion_codes': sorted(HELDOUT_CODES),
        'site_overlap_count': int(len(site_overlap)),
        'site_overlap_examples': site_overlap[:25],
        'exact_positive_pair_overlap_count': int(len(exact_positive_overlap)),
        'merops_parse_method': MEROPS_PARSE_METHOD,
        'merops_parsing_version': MEROPS_PARSING_VERSION,
        'merops_parse_examples': merops_parse_examples(positive_rows),
        'code_table_rows': int(len(code_table)),
        'code_abundance_distribution': code_table['abundance_bin'].value_counts().sort_index().to_dict(),
        'merops_family_distribution_valid_codes': code_table[code_table['protease_code'].isin(valid_codes)]['merops_family'].value_counts().sort_index().to_dict(),
        'merops_class_family_distribution': class_family.to_dict(orient='records'),
    }
    return train_positive, valid_positive, metadata


In [8]:
def blocked_pair(code, site) -> bool:
    return normalize_pair(code, site) in GLOBAL_KNOWN_POSITIVE_PAIRS


def random_site_like(length: int, rng: np.random.Generator) -> str:
    return ''.join(rng.choice(AMINO_ACIDS, size=max(1, length)).tolist())


def scramble_site_for_code(code: str, site: str, rng: np.random.Generator) -> tuple[str | None, bool]:
    chars = list(clean_site(site))
    if len(chars) < 2:
        for _ in range(100):
            candidate = random_site_like(8, rng)
            if not blocked_pair(code, candidate):
                return candidate, False
        return None, True
    original = ''.join(chars)
    for _ in range(100):
        rng.shuffle(chars)
        scrambled = ''.join(chars)
        if scrambled != original and not blocked_pair(code, scrambled):
            return scrambled, False
    for _ in range(100):
        candidate = random_site_like(len(original), rng)
        if not blocked_pair(code, candidate):
            return candidate, False
    return None, True


def candidate_pool_for_strategy(row: pd.Series, candidates: pd.DataFrame, strategy: str) -> pd.DataFrame:
    if strategy == 'random_site':
        return candidates
    if strategy == 'cross_family':
        return candidates[candidates['merops_family'] != row['merops_family']]
    raise ValueError(f'No candidate pool for strategy {strategy}')


def choose_site_from_pool(row: pd.Series, pool: pd.DataFrame, rng: np.random.Generator) -> tuple[str | None, bool]:
    if pool.empty:
        return None, True
    code_value = row['protease_code']
    indices = rng.integers(0, len(pool), size=min(200, max(20, len(pool) * 2)))
    for idx in indices:
        candidate = pool.iloc[int(idx)]
        site = candidate['site']
        if not blocked_pair(code_value, site):
            return site, False
    return None, True


def deduplicate_pairs(frame: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    before = len(frame)
    deduped = frame.drop_duplicates(subset=['protease_code', 'site', 'label']).reset_index(drop=True)
    return deduped, before - len(deduped)


def make_binary_frame(
    positives: pd.DataFrame,
    negative_strategy: str,
    negatives_per_positive: int,
    seed: int,
    partition: str,
) -> tuple[pd.DataFrame, dict]:
    positives = positives.dropna(subset=['protease_code', 'site', 'protease']).copy().reset_index(drop=True)
    positives['label'] = 1
    positives['pair_type'] = 'positive_observed'
    positives['source_positive_site'] = positives['site']
    positives['partition'] = partition
    positives['negative_strategy_used'] = ''

    rng = np.random.default_rng(seed)
    strategies = ['scramble', 'random_site', 'cross_family'] if negative_strategy == 'mixed' else [negative_strategy]
    negatives = []
    failed_negative_sampling = 0
    positives_with_no_valid_negative = 0
    repeated_negative_site_count = 0
    emitted_negative_sites = set()
    pool_sizes = []
    cross_family_pool_sizes = []
    families_in_partition = int(positives['merops_family'].nunique())
    sampling_with_replacement = True

    for idx, row in positives.iterrows():
        made_for_positive = 0
        for neg_idx in range(negatives_per_positive):
            strategy = strategies[(idx + neg_idx) % len(strategies)]
            failed = False
            if strategy == 'scramble':
                negative_site, failed = scramble_site_for_code(row['protease_code'], row['site'], rng)
                pool_sizes.append(np.nan)
            elif strategy in {'random_site', 'cross_family'}:
                pool = candidate_pool_for_strategy(row, positives, strategy)
                pool_sizes.append(int(len(pool)))
                if strategy == 'cross_family':
                    cross_family_pool_sizes.append(int(len(pool)))
                negative_site, failed = choose_site_from_pool(row, pool, rng)
            else:
                raise ValueError(f'Unknown negative strategy: {strategy}')

            if failed or negative_site is None or blocked_pair(row['protease_code'], negative_site):
                failed_negative_sampling += 1
                continue
            clean_negative_site = clean_site(negative_site)
            if clean_negative_site in emitted_negative_sites:
                repeated_negative_site_count += 1
            emitted_negative_sites.add(clean_negative_site)
            neg = row.copy()
            neg['site'] = clean_negative_site
            neg['label'] = 0
            neg['pair_type'] = f'negative_{strategy}'
            neg['source_positive_site'] = row['site']
            neg['partition'] = partition
            neg['negative_strategy_used'] = strategy
            negatives.append(neg)
            made_for_positive += 1
        if made_for_positive == 0:
            positives_with_no_valid_negative += 1

    binary = pd.concat([positives, pd.DataFrame(negatives)], ignore_index=True)
    binary, duplicates_removed = deduplicate_pairs(binary)
    binary = binary.sample(frac=1, random_state=seed).reset_index(drop=True)

    positive_count = int((binary['label'] == 1).sum())
    negative_count = int((binary['label'] == 0).sum())
    ratio = float(positive_count / negative_count) if negative_count else float('inf')
    diagnostics = {
        'partition': partition,
        'negative_strategy': negative_strategy,
        'negative_generation_version': NEGATIVE_GENERATION_VERSION,
        'pair_generation_seed': seed,
        'families_in_partition': families_in_partition,
        'positive_rows_input': int(len(positives)),
        'negative_rows_output': negative_count,
        'positive_rows_output': positive_count,
        'final_pairs': int(len(binary)),
        'duplicate_pairs_removed': int(duplicates_removed),
        'negative_candidate_pool_size_min': int(np.nanmin(pool_sizes)) if pool_sizes and not np.all(pd.isna(pool_sizes)) else None,
        'negative_candidate_pool_size_max': int(np.nanmax(pool_sizes)) if pool_sizes and not np.all(pd.isna(pool_sizes)) else None,
        'negative_candidate_pool_size_mean': float(np.nanmean(pool_sizes)) if pool_sizes and not np.all(pd.isna(pool_sizes)) else None,
        'cross_family_candidate_pool_size_min': int(np.min(cross_family_pool_sizes)) if cross_family_pool_sizes else None,
        'cross_family_candidate_pool_size_max': int(np.max(cross_family_pool_sizes)) if cross_family_pool_sizes else None,
        'cross_family_candidate_pool_size_mean': float(np.mean(cross_family_pool_sizes)) if cross_family_pool_sizes else None,
        'failed_negative_sampling_count': int(failed_negative_sampling),
        'positives_with_no_valid_negative': int(positives_with_no_valid_negative),
        'repeated_negative_site_count': int(repeated_negative_site_count),
        'sampling_with_replacement': sampling_with_replacement,
        'final_positive_to_negative_ratio': ratio,
        'final_class_balance': {'positive': positive_count, 'negative': negative_count},
    }
    if negative_strategy == 'cross_family' and families_in_partition < 2:
        diagnostics['warning'] = 'Cross-family negatives requested with fewer than two MEROPS families in partition.'
    return binary, diagnostics


PAIR_MANIFEST_COLUMNS = [
    'partition', 'label', 'protease_code', 'merops_class', 'merops_family', 'site',
    'source_positive_site', 'pair_type', 'negative_strategy_used', 'source_row_id',
    'cleavage_type', 'source_positive_count', 'protease',
]


def save_pair_manifest(frame: pd.DataFrame, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    cols = [col for col in PAIR_MANIFEST_COLUMNS if col in frame.columns]
    frame[cols].sort_values(cols[:6]).to_csv(path, index=False)
    return file_sha256(path)


def save_code_list(codes: list[str], path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'protease_code': sorted(codes)}).to_csv(path, index=False)
    return file_sha256(path)


def prepare_data_artifacts(config) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    positive_rows, source_info = load_positive_source(config.clustering, seed=config.split_seed)
    train_positive, valid_positive, split_metadata = split_codes(
        positive_rows,
        config.validation_split_strategy,
        config.split_seed,
    )
    train_pairs, train_negative_diagnostics = make_binary_frame(
        train_positive,
        config.negative_strategy,
        NEGATIVES_PER_POSITIVE,
        config.pair_generation_seed,
        'train',
    )
    valid_pairs, valid_negative_diagnostics = make_binary_frame(
        valid_positive,
        config.negative_strategy,
        NEGATIVES_PER_POSITIVE,
        config.pair_generation_seed + 1,
        'valid',
    )

    train_codes = set(train_positive['protease_code'])
    valid_codes = set(valid_positive['protease_code'])
    assert train_codes.isdisjoint(valid_codes), 'Train and validation protease codes overlap.'
    assert train_codes.isdisjoint(HELDOUT_CODES), 'Held-out code found in training.'
    assert valid_codes.isdisjoint(HELDOUT_CODES), 'Held-out code found in validation.'

    train_pairs_set = set(zip(train_pairs['protease_code'], train_pairs['site']))
    valid_pairs_set = set(zip(valid_pairs['protease_code'], valid_pairs['site']))
    exact_pair_overlap = train_pairs_set.intersection(valid_pairs_set)
    assert not exact_pair_overlap, 'Exact train/validation protease-site pairs overlap after negative generation.'

    manifest_key = short_hash({
        'pipeline_version': PIPELINE_VERSION,
        'merops_parsing_version': MEROPS_PARSING_VERSION,
        'negative_generation_version': NEGATIVE_GENERATION_VERSION,
        'clustering': config.clustering,
        'negative_strategy': config.negative_strategy,
        'validation_split_strategy': config.validation_split_strategy,
        'split_seed': config.split_seed,
        'pair_generation_seed': config.pair_generation_seed,
        'smoke_test': SMOKE_TEST,
        'dataset_hashes': DATASET_HASHES,
    }, length=16)
    manifest_dir = MANIFEST_DIR / manifest_key
    manifest_dir.mkdir(parents=True, exist_ok=True)

    train_pair_manifest = manifest_dir / 'train_pair_manifest.csv'
    valid_pair_manifest = manifest_dir / 'valid_pair_manifest.csv'
    train_pair_hash = save_pair_manifest(train_pairs, train_pair_manifest)
    valid_pair_hash = save_pair_manifest(valid_pairs, valid_pair_manifest)
    train_code_hash = save_code_list(split_metadata['train_codes'], manifest_dir / 'train_protease_codes.csv')
    valid_code_hash = save_code_list(split_metadata['valid_codes'], manifest_dir / 'valid_protease_codes.csv')
    heldout_code_hash = save_code_list(split_metadata['heldout_exclusion_codes'], manifest_dir / 'heldout_exclusion_codes.csv')

    split_manifest = {
        **split_metadata,
        'source_info': source_info,
        'train_negative_diagnostics': train_negative_diagnostics,
        'valid_negative_diagnostics': valid_negative_diagnostics,
        'train_pair_manifest': str(train_pair_manifest),
        'valid_pair_manifest': str(valid_pair_manifest),
        'train_pair_manifest_hash': train_pair_hash,
        'valid_pair_manifest_hash': valid_pair_hash,
        'train_code_list_hash': train_code_hash,
        'valid_code_list_hash': valid_code_hash,
        'heldout_code_list_hash': heldout_code_hash,
        'manifest_key': manifest_key,
        'manifest_dir': str(manifest_dir),
        'dataset_hashes': DATASET_HASHES,
        'exact_binary_pair_overlap_count': int(len(exact_pair_overlap)),
        'positive_negative_counts': {
            'train_positive_pairs': int((train_pairs['label'] == 1).sum()),
            'train_negative_pairs': int((train_pairs['label'] == 0).sum()),
            'valid_positive_pairs': int((valid_pairs['label'] == 1).sum()),
            'valid_negative_pairs': int((valid_pairs['label'] == 0).sum()),
        },
    }
    write_json(manifest_dir / 'split_metadata.json', split_manifest)
    pd.DataFrame(split_manifest['merops_class_family_distribution']).to_csv(manifest_dir / 'merops_class_family_distribution.csv', index=False)
    return train_pairs, valid_pairs, split_manifest


In [9]:
if not TORCH_AVAILABLE:
    print('Torch is unavailable; model and DataLoader classes are disabled until this notebook runs in a PyTorch/Colab environment.')
else:
    class BinaryProteinPairDataset(Dataset):
        def __init__(self, frame: pd.DataFrame, tokenizer, max_len_protease=1022, max_len_site=16):
            self.frame = frame.reset_index(drop=True)
            self.tokenizer = tokenizer
            self.max_len_protease = max_len_protease
            self.max_len_site = max_len_site

        def __len__(self):
            return len(self.frame)

        def encode(self, seq: str, max_len: int):
            return self.tokenizer(
                clean_for_esm(seq),
                padding='max_length',
                truncation=True,
                max_length=max_len,
                return_tensors='pt',
            )

        def __getitem__(self, idx):
            row = self.frame.iloc[idx]
            protease = self.encode(row['protease'], self.max_len_protease)
            site = self.encode(row['site'], self.max_len_site)
            return {
                'protease': {k: v.squeeze(0) for k, v in protease.items()},
                'site': {k: v.squeeze(0) for k, v in site.items()},
                'label': torch.tensor(row['label'], dtype=torch.float32),
            }


    TOKENIZER_CACHE = {}
    FROZEN_ENCODER_CACHE = {}


    def require_training_stack():
        if not TORCH_AVAILABLE:
            raise RuntimeError(f'Torch is not available in this runtime: {TORCH_IMPORT_ERROR}')
        if not TRANSFORMERS_AVAILABLE:
            raise RuntimeError(f'Transformers is not available in this runtime: {TRANSFORMERS_IMPORT_ERROR}')


    def get_tokenizer(model_name: str):
        require_training_stack()
        if model_name not in TOKENIZER_CACHE:
            TOKENIZER_CACHE[model_name] = AutoTokenizer.from_pretrained(model_name)
        return TOKENIZER_CACHE[model_name]


    def get_frozen_encoder(model_name: str):
        require_training_stack()
        if CACHE_FROZEN_ENCODERS and model_name in FROZEN_ENCODER_CACHE:
            return FROZEN_ENCODER_CACHE[model_name]
        encoder = AutoModel.from_pretrained(model_name)
        for param in encoder.parameters():
            param.requires_grad = False
        encoder.eval()
        if CACHE_FROZEN_ENCODERS:
            FROZEN_ENCODER_CACHE[model_name] = encoder
        return encoder


    class BinaryESMClassifier(nn.Module):
        def __init__(self, model_name: str, freeze_esm: bool, hidden_dim: int, dropout: float):
            super().__init__()
            self.model_name = model_name
            self.freeze_esm = freeze_esm
            self.encoder = get_frozen_encoder(model_name) if freeze_esm else AutoModel.from_pretrained(model_name)
            hidden_size = self.encoder.config.hidden_size
            if freeze_esm:
                for param in self.encoder.parameters():
                    param.requires_grad = False
                self.encoder.eval()
            self.classifier = nn.Sequential(
                nn.Linear(hidden_size * 4, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1),
            )

        @staticmethod
        def mean_pool(last_hidden_state, attention_mask):
            mask = attention_mask.unsqueeze(-1).float()
            summed = (last_hidden_state * mask).sum(dim=1)
            denom = mask.sum(dim=1).clamp(min=1e-9)
            return summed / denom

        def train(self, mode: bool = True):
            super().train(mode)
            self.classifier.train(mode)
            if self.freeze_esm:
                self.encoder.eval()
            return self

        def encode(self, batch):
            if self.freeze_esm:
                self.encoder.eval()
                with torch.no_grad():
                    outputs = self.encoder(**batch)
                    pooled = self.mean_pool(outputs.last_hidden_state, batch['attention_mask'])
            else:
                outputs = self.encoder(**batch)
                pooled = self.mean_pool(outputs.last_hidden_state, batch['attention_mask'])
            return F.normalize(pooled, p=2, dim=-1)

        def forward(self, protease_batch, site_batch):
            protease_emb = self.encode(protease_batch)
            site_emb = self.encode(site_batch)
            features = torch.cat([
                protease_emb,
                site_emb,
                torch.abs(protease_emb - site_emb),
                protease_emb * site_emb,
            ], dim=-1)
            return self.classifier(features).squeeze(-1)


    def get_device():
        require_training_stack()
        if torch.cuda.is_available():
            return torch.device('cuda')
        if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            return torch.device('mps')
        return torch.device('cpu')


    def move_inputs(inputs, device):
        return {k: v.to(device) for k, v in inputs.items()}


    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        encoder_total = sum(p.numel() for p in model.encoder.parameters())
        encoder_trainable = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
        return {
            'total': int(total),
            'trainable': int(trainable),
            'frozen': int(total - trainable),
            'encoder_total': int(encoder_total),
            'encoder_trainable': int(encoder_trainable),
            'classifier_trainable': int(trainable - encoder_trainable),
        }


    def build_loaders(train_pairs, valid_pairs, tokenizer, batch_size: int):
        train_dataset = BinaryProteinPairDataset(train_pairs, tokenizer, MAX_LEN_PROTEASE, MAX_LEN_SITE)
        valid_dataset = BinaryProteinPairDataset(valid_pairs, tokenizer, MAX_LEN_PROTEASE, MAX_LEN_SITE)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
        return train_loader, valid_loader


    def optimizer_parameters(model):
        return [p for p in model.parameters() if p.requires_grad]


In [10]:
def metrics_at_threshold(labels, probs, threshold):
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs).astype(float)
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    row = {
        'threshold': float(threshold),
        'mcc': float(matthews_corrcoef(labels, preds)),
        'precision': float(precision_score(labels, preds, zero_division=0)),
        'recall': float(recall_score(labels, preds, zero_division=0)),
        'f1': float(f1_score(labels, preds, zero_division=0)),
        'accuracy': float(accuracy_score(labels, preds)),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }
    try:
        row['roc_auc'] = float(roc_auc_score(labels, probs))
    except ValueError:
        row['roc_auc'] = float('nan')
    return row


def select_validation_metrics(labels, probs):
    rows = [metrics_at_threshold(labels, probs, threshold) for threshold in THRESHOLDS]
    return max(rows, key=lambda row: row['mcc'])


def metrics_with_fixed_threshold(labels, probs, threshold):
    return metrics_at_threshold(labels, probs, threshold)


def add_loss(metrics: dict, losses: list) -> dict:
    metrics = metrics.copy()
    metrics['loss'] = float(np.mean(losses)) if losses else float('nan')
    return metrics


def autocast_context(actual_use_amp: bool):
    if actual_use_amp and torch.cuda.is_available():
        return torch.cuda.amp.autocast(enabled=True)
    return nullcontext()


@torch.no_grad() if TORCH_AVAILABLE else (lambda f: f)
def predict_model(model, loader, device, loss_fn=None, actual_use_amp: bool = False):
    model.eval()
    all_probs = []
    all_labels = []
    losses = []
    for batch in tqdm(loader, desc='eval', leave=False):
        protease = move_inputs(batch['protease'], device)
        site = move_inputs(batch['site'], device)
        labels = batch['label'].to(device)
        with autocast_context(actual_use_amp):
            logits = model(protease, site)
            if loss_fn is not None:
                losses.append(loss_fn(logits, labels).item())
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())
    return np.asarray(all_labels), np.asarray(all_probs), losses


def train_one_epoch(model, loader, optimizer, device, loss_fn, gradient_accumulation_steps: int, actual_use_amp: bool):
    model.train()
    running = []
    updates = 0
    scaler = torch.cuda.amp.GradScaler(enabled=actual_use_amp)
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc='train')
    total_batches = len(loader)
    for step, batch in enumerate(pbar, start=1):
        protease = move_inputs(batch['protease'], device)
        site = move_inputs(batch['site'], device)
        labels = batch['label'].to(device)
        is_update_step = (step % gradient_accumulation_steps == 0) or (step == total_batches)
        with autocast_context(actual_use_amp):
            logits = model(protease, site)
            raw_loss = loss_fn(logits, labels)
            loss = raw_loss / gradient_accumulation_steps
        if actual_use_amp:
            scaler.scale(loss).backward()
            if is_update_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                updates += 1
        else:
            loss.backward()
            if is_update_step:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                updates += 1
        running.append(raw_loss.item())
        pbar.set_postfix(loss=f'{np.mean(running):.4f}', updates=updates)
    return float(np.mean(running)), updates


def prediction_frame(frame: pd.DataFrame, probs, threshold: float) -> pd.DataFrame:
    columns = ['partition', 'protease_code', 'merops_class', 'merops_family', 'site', 'source_positive_site', 'pair_type', 'label']
    out = frame[[col for col in columns if col in frame.columns]].copy()
    out['probability'] = probs
    out['threshold'] = threshold
    out['prediction'] = (out['probability'] >= threshold).astype(int)
    return out


def metric_value(value):
    if isinstance(value, float):
        if math.isnan(value):
            return 'nan'
        return f'{value:.4f}'
    return str(value)


def markdown_table(rows, columns):
    header = '| ' + ' | '.join(columns) + ' |'
    divider = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    body = []
    for row in rows:
        body.append('| ' + ' | '.join(metric_value(row.get(col, '')) for col in columns) + ' |')
    return '\n'.join([header, divider] + body)


In [11]:
@dataclass(frozen=True)
class ExperimentConfig:
    stage: str
    clustering: str
    negative_strategy: str
    freeze_esm: bool
    model_name: str
    learning_rate: float
    validation_split_strategy: str
    split_seed: int
    training_seed: int
    pair_generation_seed: int
    batch_size: int
    gradient_accumulation_steps: int
    use_amp: bool
    epochs: int

    @property
    def effective_batch_size(self) -> int:
        return int(self.batch_size * self.gradient_accumulation_steps)


def architecture_name(freeze_esm: bool) -> str:
    return 'frozen_esm' if freeze_esm else 'unfrozen_esm'


def model_short_name(model_name: str) -> str:
    return MODEL_SPECS.get(model_name, {}).get('short_name', re.sub(r'[^A-Za-z0-9]+', '_', model_name).strip('_').lower())


def run_mode_for_config(config: ExperimentConfig) -> str:
    if config.stage == 'esm2_150m_check':
        return 'frozen_150m_20_epoch'
    return 'standard'


def config_dict(config: ExperimentConfig) -> dict:
    data = asdict(config)
    data['architecture'] = architecture_name(config.freeze_esm)
    data['model_short_name'] = model_short_name(config.model_name)
    data['effective_batch_size'] = config.effective_batch_size
    data['pipeline_version'] = PIPELINE_VERSION
    data['merops_parsing_version'] = MEROPS_PARSING_VERSION
    data['negative_generation_version'] = NEGATIVE_GENERATION_VERSION
    data['smoke_test'] = SMOKE_TEST
    return data


def run_prefix(config: ExperimentConfig) -> str:
    return f'{config.stage}_{model_short_name(config.model_name)}_{"frozen" if config.freeze_esm else "unfrozen"}'


def run_id_for_config(config: ExperimentConfig) -> str:
    digest = short_hash(config_dict(config), length=8)
    return f'{run_prefix(config)}_{digest}'


def full_fingerprint(config: ExperimentConfig, split_manifest: dict) -> dict:
    return {
        'pipeline_version': PIPELINE_VERSION,
        'merops_parsing_version': MEROPS_PARSING_VERSION,
        'negative_generation_version': NEGATIVE_GENERATION_VERSION,
        'git_commit_hash': git_commit_hash(),
        'config': config_dict(config),
        'dataset_hashes': DATASET_HASHES,
        'split_manifest_hash': short_hash(split_manifest, length=16),
        'split_manifest_key': split_manifest['manifest_key'],
        'train_pair_manifest_hash': split_manifest['train_pair_manifest_hash'],
        'valid_pair_manifest_hash': split_manifest['valid_pair_manifest_hash'],
        'train_code_list_hash': split_manifest['train_code_list_hash'],
        'valid_code_list_hash': split_manifest['valid_code_list_hash'],
        'heldout_code_list_hash': split_manifest['heldout_code_list_hash'],
    }


def load_reusable_summary_if_available(run_dir: Path, fingerprint: dict) -> dict | None:
    summary_path = run_dir / 'run_summary.json'
    fingerprint_path = run_dir / 'fingerprint.json'
    if not run_dir.exists():
        return None
    if not summary_path.exists() or not fingerprint_path.exists():
        if ALLOW_OVERWRITE_RUNS:
            return None
        raise FileExistsError(f'Run directory exists but is incomplete: {run_dir}')
    saved_fingerprint = json.loads(fingerprint_path.read_text(encoding='utf-8'))
    if saved_fingerprint == make_json_safe(fingerprint):
        if REUSE_EXISTING_COMPLETED_RUNS:
            print(f'Reusing completed run: {run_dir}')
            return json.loads(summary_path.read_text(encoding='utf-8'))
        if not ALLOW_OVERWRITE_RUNS:
            raise FileExistsError(f'Run already exists and reuse is disabled: {run_dir}')
    if not ALLOW_OVERWRITE_RUNS:
        raise FileExistsError(f'Run directory exists with a different fingerprint: {run_dir}')
    return None


def dataset_size_info(train_pairs: pd.DataFrame, valid_pairs: pd.DataFrame, split_manifest: dict) -> dict:
    return {
        'train_positives': int((train_pairs['label'] == 1).sum()),
        'train_negatives': int((train_pairs['label'] == 0).sum()),
        'train_pairs': int(len(train_pairs)),
        'train_unique_codes': int(train_pairs['protease_code'].nunique()),
        'train_unique_sites': int(train_pairs['site'].nunique()),
        'valid_positives': int((valid_pairs['label'] == 1).sum()),
        'valid_negatives': int((valid_pairs['label'] == 0).sum()),
        'valid_pairs': int(len(valid_pairs)),
        'valid_unique_codes': int(valid_pairs['protease_code'].nunique()),
        'valid_unique_sites': int(valid_pairs['site'].nunique()),
        'site_overlap_count': split_manifest['site_overlap_count'],
    }


def save_run_summary(run_dir: Path, summary: dict, history_df: pd.DataFrame):
    write_json(run_dir / 'run_summary.json', summary)
    history_df.to_csv(run_dir / 'training_history.csv', index=False)
    metric_columns = ['loss', 'threshold', 'mcc', 'precision', 'recall', 'f1', 'accuracy', 'roc_auc', 'tn', 'fp', 'fn', 'tp']
    lines = [
        '# Run Summary',
        '',
        '## Config',
        f'- Stage: `{summary["stage"]}`',
        f'- Run mode: `{summary.get("run_mode", "standard")}`',
        f'- Model: `{summary["model_name"]}`',
        f'- Architecture: `{summary["architecture"]}`',
        f'- Clustering: `{summary["clustering"]}`',
        f'- Negative strategy: `{summary["negative_strategy"]}`',
        f'- Split strategy: `{summary["validation_split_strategy"]}`',
        f'- Split seed: `{summary["split_seed"]}`',
        f'- Training seed: `{summary["training_seed"]}`',
        f'- Pair generation seed: `{summary["pair_generation_seed"]}`',
        f'- Learning rate: `{summary["learning_rate"]}`',
        f'- Physical batch size: `{summary["batch_size"]}`',
        f'- Gradient accumulation steps: `{summary["gradient_accumulation_steps"]}`',
        f'- Effective batch size: `{summary["effective_batch_size"]}`',
        f'- Requested AMP: `{summary["requested_amp"]}`',
        f'- Actual AMP: `{summary["actual_amp"]}`',
        '',
        '## Validation',
        markdown_table([summary['validation_metrics']], metric_columns),
        '',
        '## Dataset Sizes',
        markdown_table([summary['dataset_size']], sorted(summary['dataset_size'].keys())),
        '',
        'Held-out metrics are not included in intermediate run summaries.',
        '',
    ]
    if summary.get('warnings'):
        lines.extend(['## Warnings'] + [f'- {warning}' for warning in summary['warnings']] + [''])
    (run_dir / 'run_summary.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')


In [12]:
def evaluate_pairs(model, frame: pd.DataFrame, tokenizer, device, loss_fn, threshold: float, batch_size: int, actual_use_amp: bool):
    dataset = BinaryProteinPairDataset(frame, tokenizer, MAX_LEN_PROTEASE, MAX_LEN_SITE)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    labels, probs, losses = predict_model(model, loader, device, loss_fn, actual_use_amp=actual_use_amp)
    metrics = add_loss(metrics_with_fixed_threshold(labels, probs, threshold), losses)
    preds = prediction_frame(frame, probs, threshold)
    return metrics, preds


def train_experiment(config: ExperimentConfig) -> dict:
    if RUN_HELDOUT_EVALUATION:
        raise ValueError('RUN_HELDOUT_EVALUATION must be False for intermediate training runs.')
    set_seed(config.training_seed)
    train_pairs, valid_pairs, split_manifest = prepare_data_artifacts(config)
    fingerprint = full_fingerprint(config, split_manifest)
    run_id = run_id_for_config(config)
    run_dir = OUTPUT_DIR / config.stage / run_id
    reusable = load_reusable_summary_if_available(run_dir, fingerprint)
    if reusable is not None:
        return reusable
    run_dir.mkdir(parents=True, exist_ok=True)
    write_json(run_dir / 'config.json', config_dict(config))
    write_json(run_dir / 'fingerprint.json', fingerprint)
    write_json(run_dir / 'split_metadata.json', split_manifest)

    device = get_device()
    actual_amp = bool(config.use_amp and device.type == 'cuda')
    tokenizer = get_tokenizer(config.model_name)
    train_loader, valid_loader = build_loaders(train_pairs, valid_pairs, tokenizer, config.batch_size)
    model = BinaryESMClassifier(config.model_name, config.freeze_esm, HIDDEN_DIM, DROPOUT).to(device)
    if config.freeze_esm:
        model.encoder.eval()
    parameter_summary = count_parameters(model)
    optimizer = AdamW(optimizer_parameters(model), lr=config.learning_rate, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()
    history = []
    best_mcc = -1.0
    best_threshold = 0.5
    best_state = None
    checkpoint_path = run_dir / 'best_model.pt'
    optimizer_updates_per_epoch = []

    print(
        f'Run: {config.stage} | {config.validation_split_strategy} | {config.model_name} | '
        f'{architecture_name(config.freeze_esm)} | lr={config.learning_rate} | '
        f'split_seed={config.split_seed} | training_seed={config.training_seed}'
    )
    for epoch in range(1, config.epochs + 1):
        train_loss, updates = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device,
            loss_fn,
            config.gradient_accumulation_steps,
            actual_amp,
        )
        optimizer_updates_per_epoch.append(updates)
        labels, probs, losses = predict_model(model, valid_loader, device, loss_fn, actual_use_amp=actual_amp)
        valid_metrics = add_loss(select_validation_metrics(labels, probs), losses)
        valid_metrics['epoch'] = epoch
        valid_metrics['train_loss'] = train_loss
        valid_metrics['optimizer_updates'] = updates
        history.append(valid_metrics)
        print(f'Epoch {epoch}: valid_mcc={valid_metrics["mcc"]:.4f}, threshold={valid_metrics["threshold"]:.2f}, updates={updates}')
        if valid_metrics['mcc'] > best_mcc:
            best_mcc = valid_metrics['mcc']
            best_threshold = valid_metrics['threshold']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': config_dict(config),
                'threshold': best_threshold,
                'validation_metrics': valid_metrics,
                'parameter_summary': parameter_summary,
                'fingerprint': make_json_safe(fingerprint),
            }, checkpoint_path)

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    history_df = pd.DataFrame(history)
    labels, probs, losses = predict_model(model, valid_loader, device, loss_fn, actual_use_amp=actual_amp)
    final_valid_metrics = add_loss(metrics_with_fixed_threshold(labels, probs, best_threshold), losses)
    valid_predictions = prediction_frame(valid_pairs, probs, best_threshold)
    valid_predictions.to_csv(run_dir / 'validation_predictions.csv', index=False)

    warnings_list = []
    if split_manifest['n_valid_codes'] < MIN_VALID_CODES_WARNING:
        warnings_list.append(f'Validation has only {split_manifest["n_valid_codes"]} protease codes.')
    if split_manifest['n_valid_positive_rows'] < MIN_VALID_POSITIVES_WARNING:
        warnings_list.append(f'Validation has only {split_manifest["n_valid_positive_rows"]} positive examples.')

    summary = {
        **config_dict(config),
        'run_id': run_id,
        'run_dir': str(run_dir),
        'stage': config.stage,
        'clustering': config.clustering,
        'negative_strategy': config.negative_strategy,
        'freeze_esm': config.freeze_esm,
        'architecture': architecture_name(config.freeze_esm),
        'model_name': config.model_name,
        'learning_rate': config.learning_rate,
        'validation_split_strategy': config.validation_split_strategy,
        'split_seed': config.split_seed,
        'training_seed': config.training_seed,
        'pair_generation_seed': config.pair_generation_seed,
        'batch_size': config.batch_size,
        'gradient_accumulation_steps': config.gradient_accumulation_steps,
        'effective_batch_size': config.effective_batch_size,
        'epochs': config.epochs,
        'optimizer_updates_per_epoch': optimizer_updates_per_epoch,
        'requested_amp': bool(config.use_amp),
        'actual_amp': actual_amp,
        'run_mode': run_mode_for_config(config),
        'selection_metric': 'validation_mcc',
        'validation_threshold': best_threshold,
        'validation_metrics': final_valid_metrics,
        'dataset_size': dataset_size_info(train_pairs, valid_pairs, split_manifest),
        'warnings': warnings_list,
        'parameter_summary': parameter_summary,
        'checkpoint_path': str(checkpoint_path),
        'heldout_used_for_selection': False,
        'heldout_metrics_present': False,
        'split_manifest_key': split_manifest['manifest_key'],
        'split_manifest_hash': fingerprint['split_manifest_hash'],
        'train_pair_manifest_hash': split_manifest['train_pair_manifest_hash'],
        'valid_pair_manifest_hash': split_manifest['valid_pair_manifest_hash'],
        'fingerprint_hash': short_hash(fingerprint, length=16),
    }
    save_run_summary(run_dir, summary, history_df)

    del model
    gc.collect()
    if TORCH_AVAILABLE and torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary


## Optional ESM2 150M larger-model check

 The goal is to test whether a larger ESM2 backbone gives a useful model-capacity trend after the completed frozen 8M and frozen 35M checks.

Keep the data setup fixed: `no_clustering`, `cross_family`, `code_group_random`, split seed `42`, training seed `42`, and pair-generation seed `4242`. The intended change is model size only: ESM2 150M instead of ESM2 8M or 35M.

The stage runs frozen ESM2 150M for 20 epochs. This is the practical report-relevant check because unfrozen 150M fine-tuning was too slow on Colab, while 650M was discussed as likely too large. Results can be compared cautiously with the frozen 8M and frozen 35M model-capacity runs because the data setup and effective batch size are kept aligned.

Held-out evaluation remains disabled. Do not use held-out metrics for this stage unless a final configuration is selected later using validation evidence only.


In [13]:
@dataclass(frozen=True)
class StageConfigFactory:
    training_seed: int = TRAINING_SEED
    pair_generation_seed: int = PAIR_GENERATION_SEED


def make_config(
    stage: str,
    model_name: str,
    freeze_esm: bool,
    learning_rate: float,
    validation_split_strategy: str,
    split_seed: int,
    training_seed: int = TRAINING_SEED,
    pair_generation_seed: int = PAIR_GENERATION_SEED,
    clustering: str = 'no_clustering',
    negative_strategy: str = 'cross_family',
    use_amp: bool = True,
    epochs: int = EPOCHS,
) -> ExperimentConfig:
    spec = MODEL_SPECS[model_name]
    if freeze_esm:
        batch_size = spec['frozen_batch_size']
        grad_accum = spec['frozen_grad_accum']
    else:
        batch_size = spec['unfrozen_batch_size']
        grad_accum = spec['unfrozen_grad_accum']
    return ExperimentConfig(
        stage=stage,
        clustering=clustering,
        negative_strategy=negative_strategy,
        freeze_esm=freeze_esm,
        model_name=model_name,
        learning_rate=learning_rate,
        validation_split_strategy=validation_split_strategy,
        split_seed=split_seed,
        training_seed=training_seed,
        pair_generation_seed=pair_generation_seed,
        batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        use_amp=use_amp,
        epochs=epochs,
    )


def active_validation_check_split_seeds() -> list[int]:
    return SMOKE_VALIDATION_CHECK_SPLIT_SEEDS if SMOKE_TEST else VALIDATION_CHECK_SPLIT_SEEDS


def model_capacity_configs() -> list[ExperimentConfig]:
    return [
        make_config('model_capacity', 'facebook/esm2_t6_8M_UR50D', True, LR_FROZEN, 'code_group_random', DEFAULT_SPLIT_SEED),
        make_config('model_capacity', 'facebook/esm2_t12_35M_UR50D', True, LR_FROZEN, 'code_group_random', DEFAULT_SPLIT_SEED),
    ]


def lr_tuning_configs() -> list[ExperimentConfig]:
    learning_rates = list(LR_TUNING_VALUES)
    if INCLUDE_LR_1E4_SANITY_CHECK:
        learning_rates.append(LR_1E4_SANITY_VALUE)
    return [
        make_config('lr_tuning', 'facebook/esm2_t6_8M_UR50D', False, lr, 'code_group_random', DEFAULT_SPLIT_SEED)
        for lr in learning_rates
    ]


def validation_check_configs() -> list[ExperimentConfig]:
    strategies = ['code_group_random', 'family_stratified_code', 'family_abundance_stratified_code']
    return [
        make_config('validation_check', 'facebook/esm2_t6_8M_UR50D', True, LR_FROZEN, strategy, split_seed)
        for strategy in strategies
        for split_seed in active_validation_check_split_seeds()
    ]


def esm2_150m_check_configs() -> list[ExperimentConfig]:
    return [
        make_config(
            'esm2_150m_check',
            MODEL_150M,
            True, #frozen ESM
            lr,
            'code_group_random',
            DEFAULT_SPLIT_SEED,
            epochs=EPOCHS_150M,
        )
        for lr in LRS_150M_FROZEN
    ]


def configs_for_stage(stage: str) -> list[ExperimentConfig]:
    if stage == 'model_capacity':
        return model_capacity_configs()
    if stage == 'lr_tuning':
        return lr_tuning_configs()
    if stage == 'validation_check':
        return validation_check_configs()
    if stage == 'esm2_150m_check':
        return esm2_150m_check_configs()
    if stage == 'all':
        return model_capacity_configs() + lr_tuning_configs() + validation_check_configs()
    raise ValueError(f'Unknown RUN_STAGE: {stage}')


def index_row(summary: dict) -> dict:
    return {
        'run_id': summary['run_id'],
        'stage': summary['stage'],
        'model_name': summary['model_name'],
        'model_short_name': summary['model_short_name'],
        'architecture': summary['architecture'],
        'clustering': summary['clustering'],
        'negative_strategy': summary['negative_strategy'],
        'validation_split_strategy': summary['validation_split_strategy'],
        'split_seed': summary['split_seed'],
        'training_seed': summary['training_seed'],
        'pair_generation_seed': summary['pair_generation_seed'],
        'learning_rate': summary['learning_rate'],
        'batch_size': summary['batch_size'],
        'gradient_accumulation_steps': summary['gradient_accumulation_steps'],
        'effective_batch_size': summary['effective_batch_size'],
        'epochs': summary['epochs'],
        'run_mode': summary.get('run_mode', 'standard'),
        'requested_amp': summary['requested_amp'],
        'actual_amp': summary['actual_amp'],
        'validation_mcc': summary['validation_metrics']['mcc'],
        'validation_threshold': summary['validation_threshold'],
        'valid_unique_codes': summary['dataset_size']['valid_unique_codes'],
        'valid_positives': summary['dataset_size']['valid_positives'],
        'valid_negatives': summary['dataset_size']['valid_negatives'],
        'split_manifest_key': summary['split_manifest_key'],
        'train_pair_manifest_hash': summary['train_pair_manifest_hash'],
        'valid_pair_manifest_hash': summary['valid_pair_manifest_hash'],
        'run_dir': summary['run_dir'],
    }


def summarize_validation_check(summaries: list[dict]) -> pd.DataFrame:
    rows = []
    for summary in summaries:
        if summary['stage'] != 'validation_check':
            continue
        split_metadata_path = Path(summary['run_dir']) / 'split_metadata.json'
        split_metadata = json.loads(split_metadata_path.read_text(encoding='utf-8'))
        rows.append({
            'validation_split_strategy': summary['validation_split_strategy'],
            'split_seed': summary['split_seed'],
            'validation_mcc': summary['validation_metrics']['mcc'],
            'valid_unique_codes': summary['dataset_size']['valid_unique_codes'],
            'fallback_used': bool(split_metadata['strategy_metadata'].get('fallback_used', False)),
            'actual_split_strategy': split_metadata['actual_split_strategy'],
            'merops_family_distribution': canonical_json(split_metadata['merops_family_distribution_valid_codes']),
            'code_abundance_distribution': canonical_json(split_metadata['code_abundance_distribution']),
        })
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    summary = df.groupby('validation_split_strategy').agg(
        mean_validation_mcc=('validation_mcc', 'mean'),
        std_validation_mcc=('validation_mcc', 'std'),
        min_validation_mcc=('validation_mcc', 'min'),
        max_validation_mcc=('validation_mcc', 'max'),
        mean_valid_protease_codes=('valid_unique_codes', 'mean'),
        min_valid_protease_codes=('valid_unique_codes', 'min'),
        max_valid_protease_codes=('valid_unique_codes', 'max'),
        fallback_frequency=('fallback_used', 'mean'),
        split_seeds=('split_seed', lambda values: list(values)),
        merops_family_distributions=('merops_family_distribution', lambda values: list(values)),
        code_abundance_distributions=('code_abundance_distribution', lambda values: list(values)),
    ).reset_index()
    summary['std_validation_mcc'] = summary['std_validation_mcc'].fillna(0.0)
    summary['analysis_note'] = 'Validation sensitivity analysis only; do not select split by highest MCC.'
    summary['generalisation_note'] = 'Family-stratified splits test unseen protease codes within represented MEROPS families, not entirely unseen MEROPS families.'
    return summary


def run_configs(configs: list[ExperimentConfig]) -> list[dict]:
    summaries = []
    for config in configs:
        summary = train_experiment(config)
        summaries.append(summary)
        pd.DataFrame([index_row(s) for s in summaries]).to_csv(OUTPUT_DIR / f'{RUN_STAGE}_experiment_index.csv', index=False)
    return summaries


In [ ]:
configs = configs_for_stage(RUN_STAGE)
print(f'Prepared {len(configs)} config(s) for RUN_STAGE={RUN_STAGE!r}')
for config in configs:
    print(config_dict(config))

all_run_summaries = run_configs(configs)
index_df = pd.DataFrame([index_row(summary) for summary in all_run_summaries])
index_path = OUTPUT_DIR / f'{RUN_STAGE}_experiment_index.csv'
index_df.to_csv(index_path, index=False)
print(f'Saved experiment index: {index_path}')

if RUN_STAGE == 'validation_check' or any(summary['stage'] == 'validation_check' for summary in all_run_summaries):
    validation_summary = summarize_validation_check(all_run_summaries)
    validation_summary_path = OUTPUT_DIR / 'validation_sensitivity_summary.csv'
    validation_summary.to_csv(validation_summary_path, index=False)
    print(f'Saved validation sensitivity summary: {validation_summary_path}')
    display(validation_summary)
else:
    display(index_df)

assert not any('heldout' in col.lower() and col != 'heldout_used_for_selection' for col in index_df.columns), 'Held-out metric leaked into index table.'
print('Intermediate run complete. Held-out metrics were not calculated or displayed.')


Prepared 1 config(s) for RUN_STAGE='esm2_150m_check'
{'stage': 'esm2_150m_check', 'clustering': 'no_clustering', 'negative_strategy': 'cross_family', 'freeze_esm': True, 'model_name': 'facebook/esm2_t30_150M_UR50D', 'learning_rate': 1e-07, 'validation_split_strategy': 'code_group_random', 'split_seed': 42, 'training_seed': 42, 'pair_generation_seed': 4242, 'batch_size': 1, 'gradient_accumulation_steps': 4, 'use_amp': True, 'epochs': 9, 'architecture': 'frozen_esm', 'model_short_name': 'esm2_150m', 'effective_batch_size': 4, 'pipeline_version': 'sequential_binary_followup_v2', 'merops_parsing_version': 'merops_regex_class_family_v1', 'negative_generation_version': 'partition_local_cross_family_v1', 'smoke_test': False}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

...

eval:   0%|          | 0/12272 [00:00<?, ?it/s]

Epoch 8: valid_mcc=0.0000, threshold=0.05, updates=24974


/tmp/ipykernel_1636/3751985258.py:70: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=actual_use_amp)


train:   0%|          | 0/99894 [00:00<?, ?it/s]

/tmp/ipykernel_1636/3751985258.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast(enabled=True)


In [ ]:
# Manual held-out evaluation guard.
# This cell intentionally does nothing unless RUN_HELDOUT_EVALUATION is set to True after
# a final configuration has been selected using validation evidence only.

SELECTED_RUN_SUMMARY_PATH = OUTPUT_DIR / "lr_tuning" / "lr_tuning_esm2_8m_unfrozen_aaea37e3" / "run_summary.json"


def make_heldout_pairs(positive_frame: pd.DataFrame, negative_strategy: str, pair_generation_seed: int, partition: str):
    pairs, diagnostics = make_binary_frame(
        positive_frame,
        negative_strategy,
        NEGATIVES_PER_POSITIVE,
        pair_generation_seed,
        partition,
    )
    return pairs, diagnostics


def evaluate_heldout_from_selected_run(summary_path: Path) -> pd.DataFrame:
    if not RUN_HELDOUT_EVALUATION:
        raise ValueError('Set RUN_HELDOUT_EVALUATION=True only after validation-only final selection.')
    if summary_path is None:
        raise ValueError('SELECTED_RUN_SUMMARY_PATH must point to a validation-selected run_summary.json.')
    summary = json.loads(Path(summary_path).read_text(encoding='utf-8'))
    device = get_device()
    checkpoint = torch.load(summary['checkpoint_path'], map_location=device)
    model = BinaryESMClassifier(summary['model_name'], summary['freeze_esm'], HIDDEN_DIM, DROPOUT).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    tokenizer = get_tokenizer(summary['model_name'])
    loss_fn = nn.BCEWithLogitsLoss()
    threshold = summary['validation_threshold']
    actual_amp = bool(summary.get('actual_amp', False) and device.type == 'cuda')
    rows = []
    run_dir = Path(summary['run_dir'])
    for name, positive_frame in [
        ('test_1_sample', test_singleton_raw),
        ('test_2_to_10_samples', test_low_sample_raw),
    ]:
        heldout_pairs, diagnostics = make_heldout_pairs(
            positive_frame,
            summary['negative_strategy'],
            summary['pair_generation_seed'] + 100,
            name,
        )
        metrics, preds = evaluate_pairs(
            model,
            heldout_pairs,
            tokenizer,
            device,
            loss_fn,
            threshold,
            summary['batch_size'],
            actual_amp,
        )
        metrics['dataset'] = name
        metrics['rows'] = int(len(heldout_pairs))
        
        metrics['used_for_selection'] = False
        metrics['selected_by_validation_only'] = True
        metrics['negative_generation_diagnostics'] = diagnostics
        rows.append(metrics)
        preds.to_csv(run_dir / f'{name}_heldout_predictions.csv', index=False)
    heldout_df = pd.DataFrame(rows)
    heldout_df.to_csv(run_dir / 'manual_final_heldout_metrics.csv', index=False)
    return heldout_df


if RUN_HELDOUT_EVALUATION:
    heldout_df = evaluate_heldout_from_selected_run(Path(SELECTED_RUN_SUMMARY_PATH))
    display(heldout_df)
else:
    print('Held-out evaluation is disabled. This is expected for all intermediate experiments and smoke tests.')
